# RSNA Knee Abnormality Detection — CoAtNet 2.5-D attention-MIL (offline submission)

**Why this architecture.** Research into this competition (public write-ups, GitHub repos tracking
the public leaderboard, and this notebook's own experiment history) converged on one finding: the
strongest public solutions (~0.93-0.945 public LB) are not EfficientNet/DINO/CLIP backbones fine-tuned
end-to-end on the 58 gold-labeled studies -- that path tops out around 0.65-0.71 OOF no matter which
backbone you swap in, because 58 labels is too little signal for full fine-tuning. The winning pattern
is a **CoAtNet-based 2.5-D attention-MIL classifier**: a convolution+attention hybrid backbone (CoAtNet,
via `timm`) that reads several 2.5-D slice triplets per MRI view (sagittal/coronal/axial), pools them
with a **per-finding masked-softmax attention head** (not plain mean-pooling -- different findings
live in different slices/views, so each of the 12 targets gets its own attention distribution over
slice x view tokens), and is trained against a **blend of the 58 gold labels and weak labels mined
from the free-text reports** for the remaining ~4,300 studies. Report-derived weak supervision is what
makes a CNN/ViT trainable at all here -- without it you are training on 58 examples.

**What this notebook builds, concretely:**
1. A `timm` CoAtNet backbone (`coatnet_rmlp_2_rw_384`, the variant the strongest public checkpoints use;
   falls back to a smaller CoAtNet if that exact variant isn't available in your `timm` version).
2. A 2.5-D volume builder: each view is read as several overlapping 3-slice (2.5-D) windows, 2nd-98th
   percentile windowed, resized -- the same normalization public solutions report using.
3. A per-finding attention-MIL head pooling every slice-window x view token into one 12-way prediction,
   so each finding can attend to the view/slices that actually show it.
4. A weak-label table from the reports (keyword + simple rules -- deliberately simple and auditable;
   swap in a stronger public label table if you attach one, see the placeholder below).
5. A **checkpoint-reuse path**: if you attach other competitors' public pretrained CoAtNet checkpoints
   as a Kaggle Dataset (several exist for this competition -- see "Optional: reuse public checkpoints"
   below), the notebook loads any `.pt`/`.pth` state dict it finds, scores it standalone, and folds it
   into the final ensemble -- this is how public solutions reach 0.92+ per arm without you training from
   scratch. If none are attached, it trains this architecture itself, time-boxed, on gold+weak labels.
6. **Rank-mean ensembling** across every arm that produced valid (non-constant) predictions, with a
   0.5 benchmark fallback, so a bad/missing arm can never break the submission.

## Required input: competition data
Attach the competition dataset as usual.

## Optional (but strongly recommended) inputs -- see the step-by-step guide in the final cell
- **A public CoAtNet checkpoint dataset for this competition** (any Kaggle Dataset containing `.pt`/`.pth`
  files for a knee-MRI CoAtNet model) -- search Kaggle Datasets for this competition's name and "coatnet"
  / "raptor" / "checkpoint". Attach one or more; each usable checkpoint becomes its own ensemble arm,
  worth far more than anything this notebook can train from scratch in an hour.
- **A public report-label table** (a CSV with `StudyInstanceUID` + the 12 label columns, graded 0-1) --
  search Kaggle Datasets for this competition's name and "labels" / "weak labels". If attached, it
  replaces the notebook's simple keyword labeler as the weak-label source for training.

Both are optional placeholders: the notebook runs end-to-end and produces a valid submission with
neither attached, just with a weaker single self-trained arm instead of a multi-arm ensemble.

In [1]:
import os, re, gc, io, time, json, random, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # reduces OOM from fragmentation

import numpy as np
import pandas as pd
import pydicom
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WORK_DIR = "/kaggle/working"
LABEL_COLS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
              "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
VIEWS = ["Sagittal", "Coronal", "Axial"]

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
SEED = 42
seed_all(SEED)

GLOBAL_START = time.time()
TIME_BUDGET_SEC = 8.0 * 3600   # keep a buffer under the 9h competition limit
def time_left():
    return TIME_BUDGET_SEC - (time.time() - GLOBAL_START)
def time_left_str():
    return f"{time_left() / 60:.1f} min left"

print("Device:", DEVICE, "| CPUs:", os.cpu_count())


Device: cuda | CPUs: 4


## 1. Locate inputs (no internet, no hard-coded dataset slugs)

Walks `/kaggle/input` (skipping the huge DICOM folders) and classifies whatever is attached:
competition CSVs, any CoAtNet-looking checkpoint files, and any public weak-label CSV.

In [2]:
SKIP_DIRS = {"train_series", "test_series"}

def walk_input():
    for cur, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in SKIP_DIRS]
        yield cur, dirs, files

DATA_DIR = next((cur for cur, _, files in walk_input()
                  if "train.csv" in files and "train_series.csv" in files), None)
assert DATA_DIR, "Competition data not found under /kaggle/input -- attach the competition dataset."

# ---- optional: any public CoAtNet / knee-MRI checkpoint files (.pt / .pth) ----
CHECKPOINT_PATHS = []
for cur, _, files in walk_input():
    if "competitions" in cur or cur == DATA_DIR:
        continue
    for f in files:
        if f.endswith((".pt", ".pth")):
            CHECKPOINT_PATHS.append(os.path.join(cur, f))

# ---- optional: a public report-label CSV (StudyInstanceUID + the 12 label columns) ----
def is_label_table(path):
    try:
        head = pd.read_csv(path, nrows=5)
    except Exception:
        return False
    return "StudyInstanceUID" in head.columns and sum(c in head.columns for c in LABEL_COLS) >= 10

WEAK_LABEL_TABLE_PATH = None
for cur, _, files in walk_input():
    if cur == DATA_DIR or "competitions" in cur:
        continue
    for f in files:
        if f.endswith(".csv") and is_label_table(os.path.join(cur, f)):
            WEAK_LABEL_TABLE_PATH = os.path.join(cur, f)
            break
    if WEAK_LABEL_TABLE_PATH:
        break

# used only to seed the backbone before training a from-scratch arm (section 5/8) -- the first
# detected checkpoint, since this notebook has no internet access to fetch pretrained weights
BACKBONE_INIT_PATH = CHECKPOINT_PATHS[0] if CHECKPOINT_PATHS else None

print("DATA_DIR             :", DATA_DIR)
print("Checkpoint files found:", len(CHECKPOINT_PATHS))
for p in CHECKPOINT_PATHS:
    print("  -", p)
print("Public weak-label table:", WEAK_LABEL_TABLE_PATH or "(none attached -- will use the built-in keyword labeler)")
print("Backbone init source  :", BACKBONE_INIT_PATH or "(none -- backbone trains from random init, no internet used)")


DATA_DIR             : /kaggle/input/competitions/rsna-knee-abnormality-detection
Checkpoint files found: 1
  - /kaggle/input/datasets/amudhana/coatnet0-imagenet/coatnet_0_rw_224.pth
Public weak-label table: /kaggle/input/datasets/mammadamin/rsna-report-labels/report_labels_v2.csv
Backbone init source  : /kaggle/input/datasets/amudhana/coatnet0-imagenet/coatnet_0_rw_224.pth


## 2. Load CSVs and split gold / unlabeled studies

In [3]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
train_series = pd.read_csv(f"{DATA_DIR}/train_series.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
test_series = pd.read_csv(f"{DATA_DIR}/test_series.csv")
sample_submission = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
print(train.shape, train_series.shape, test.shape, test_series.shape)

has_labels = train[LABEL_COLS].notna().all(axis=1)
gold_df = train[has_labels].reset_index(drop=True)
unlabeled_df = train[~has_labels].reset_index(drop=True)
print(f"gold={len(gold_df)}  unlabeled={len(unlabeled_df)}")


(4407, 14) (24371, 5) (3, 1) (15, 5)
gold=58  unlabeled=4349


## 3. Weak labels for the unlabeled studies

If a public/LLM weak-label table is attached (auto-detected in section 1 -- this is exactly the format of a table like `report_labels_v2.csv`: `StudyInstanceUID` + the 12 graded 0-1 label columns, optionally with `<label>__conf` confidence and `<label>__verdict` columns alongside), it is used directly as the weak-label source, and per-label confidence (when present) is carried through as a **per-sample, per-label loss weight** in training (section 8) -- low-confidence/`UNK` rows still contribute, just less. This is a meaningfully stronger signal than this notebook's own from-scratch labeler could produce (public write-ups on this competition measure such tables at ~0.81-0.90 gold agreement, vs ~0.65 for a keyword rule-based labeler), so **attach a labeled table if you have one.**

Falls back to a transparent keyword/rule labeler (confidence fixed at 1.0) only when no such table is attached, so the notebook still runs end-to-end on just the competition data.

In [4]:
def clean_report(t):
    if not isinstance(t, str):
        return ""
    t = re.sub(r"\[.*?\]", "", t)
    return re.sub(r"\s+", " ", t).strip().lower()

KEYWORDS = {
    "ACL": [r"\bacl\b", r"anterior cruciate"],
    "MCL": [r"\bmcl\b", r"medial collateral ligament"],
    "Medial Meniscus": [r"medial meniscus"],
    "Lateral Meniscus": [r"lateral meniscus"],
    "Medial OA": [r"medial.{0,30}(osteoarthritis|compartment narrowing|joint space narrowing)"],
    "Lateral OA": [r"lateral.{0,30}(osteoarthritis|compartment narrowing|joint space narrowing)"],
    "PF OA": [r"patellofemoral.{0,30}(osteoarthritis|chondromalacia)", r"chondromalacia"],
    "Effusion": [r"\beffusion\b", r"joint fluid"],
    "Synovitis": [r"synoviti?s"],
    "Baker's": [r"baker'?s? cyst", r"popliteal cyst"],
    "Contusion": [r"contusion", r"bone bruis"],
    "Fracture": [r"fracture"],
}
NEGATION = re.compile(r"\b(no|without|absence of|negative for|not seen|no evidence of)\b")
KW_RE = {lab: re.compile("|".join(pats)) for lab, pats in KEYWORDS.items()}

def keyword_label(report):
    text = clean_report(report)
    out = {}
    for lab, rx in KW_RE.items():
        score = 0.0
        for m in rx.finditer(text):
            window = text[max(0, m.start() - 25):m.start()]
            score = max(score, 0.15 if NEGATION.search(window) else 0.85)
        out[lab] = score
    return out

CONF_COLS = [f"{lab}__conf" for lab in LABEL_COLS]

if WEAK_LABEL_TABLE_PATH:
    weak_raw = pd.read_csv(WEAK_LABEL_TABLE_PATH)
    has_conf = all(c in weak_raw.columns for c in CONF_COLS)
    keep_cols = ["StudyInstanceUID"] + LABEL_COLS + (CONF_COLS if has_conf else [])
    weak_raw = weak_raw[keep_cols].copy()
    merged = unlabeled_df[["StudyInstanceUID"]].merge(weak_raw, on="StudyInstanceUID", how="inner")
    weak_label_df = merged[["StudyInstanceUID"] + LABEL_COLS].copy()
    if has_conf:
        weak_conf_df = merged[["StudyInstanceUID"] + CONF_COLS].copy()
        weak_conf_df.columns = ["StudyInstanceUID"] + LABEL_COLS   # align names for the merge in section 7
        print(f"Using attached weak-label table WITH per-label confidence: {len(weak_label_df)} studies matched.")
    else:
        weak_conf_df = None
        print(f"Using attached weak-label table (no confidence columns found, weight=1.0): "
              f"{len(weak_label_df)} studies matched.")
else:
    records = []
    for _, row in tqdm(unlabeled_df.iterrows(), total=len(unlabeled_df), desc="keyword weak-labeling"):
        rec = keyword_label(row["Report"])
        rec["StudyInstanceUID"] = row["StudyInstanceUID"]
        records.append(rec)
    weak_label_df = pd.DataFrame(records)[["StudyInstanceUID"] + LABEL_COLS]
    weak_conf_df = None   # keyword labeler has no calibrated confidence -> uniform weight 1.0
    print(f"Built keyword weak labels for {len(weak_label_df)} studies.")

print(weak_label_df[LABEL_COLS].mean().round(3))


Using attached weak-label table WITH per-label confidence: 4349 studies matched.
ACL                 0.296
MCL                 0.213
Medial Meniscus     0.484
Lateral Meniscus    0.277
Medial OA           0.400
Lateral OA          0.321
PF OA               0.456
Effusion            0.499
Synovitis           0.339
Baker's             0.345
Contusion           0.322
Fracture            0.240
dtype: float64


## 4. Series selection + 2.5-D volume builder

For each study and each view (sagittal/coronal/axial), picks the most informative series (fluid-sensitive preferred), reads a fixed number of slices spanning the central 15-85% of the series (the part of the volume public solutions report using -- skips near-empty end slices), windows each slice to its own 2nd-98th percentile, and resizes with area interpolation.

In [5]:
N_SLICES = 5           # per view, per study -- lowered from 9 to fix CUDA OOM on a T4 (14-16 GB)
IMG_SIZE = 224        # lowered from 224 for the same reason; raise once OOM-free at these settings

def pick_view_series(rows, plane):
    r = rows[rows["Anatomical_Plane"] == plane]
    if len(r) == 0:
        return None
    f = r[r["Fluid_Sensitive"] == 1]
    return (f if len(f) else r).iloc[0]["SeriesInstanceUID"]

def sorted_files(d):
    files = [f for f in os.listdir(d) if not f.startswith(".")]
    stems = [os.path.splitext(f)[0] for f in files]
    if files and all(s.isdigit() for s in stems):
        return [f for _, f in sorted(zip(map(int, stems), files))]
    keyed = []
    for f in files:
        try:
            n = int(pydicom.dcmread(os.path.join(d, f), stop_before_pixels=True).InstanceNumber)
        except Exception:
            n = 0
        keyed.append((n, f))
    return [f for _, f in sorted(keyed)]

def load_view_slices(split, study, series, n_slices=N_SLICES, img_size=IMG_SIZE):
    d = f"{DATA_DIR}/{split}_series/{study}/{series}"
    files = sorted_files(d)
    if not files:
        return np.zeros((n_slices, img_size, img_size), np.float32)
    lo_i, hi_i = int(0.15 * len(files)), int(0.85 * len(files))
    hi_i = max(hi_i, lo_i + 1)
    idx = np.linspace(lo_i, hi_i - 1, n_slices).round().astype(int).clip(0, len(files) - 1)
    imgs = []
    for i in idx:
        try:
            a = pydicom.dcmread(os.path.join(d, files[i])).pixel_array.astype(np.float32)
            if a.ndim == 3:
                a = a[a.shape[0] // 2]
            lo, hi = np.percentile(a, [2, 98])
            a = np.clip((a - lo) / (hi - lo + 1e-6), 0, 1)
            a = cv2.resize(a, (img_size, img_size), interpolation=cv2.INTER_AREA)
        except Exception:
            a = np.zeros((img_size, img_size), np.float32)
        imgs.append(a)
    return np.stack(imgs).astype(np.float32)

def build_study_volume(split, study, rows, n_slices=N_SLICES, img_size=IMG_SIZE):
    """Returns (views, n_slices, H, W) float32 in [0,1] and a (views,) availability mask."""
    vol = np.zeros((len(VIEWS), n_slices, img_size, img_size), np.float32)
    mask = np.zeros(len(VIEWS), np.float32)
    for v, plane in enumerate(VIEWS):
        s = pick_view_series(rows, plane)
        if s is None:
            continue
        try:
            vol[v] = load_view_slices(split, study, s, n_slices, img_size)
            mask[v] = 1.0
        except Exception:
            pass
    if mask.sum() == 0:
        mask[0] = 1.0
    return vol, mask


## 5. CoAtNet 2.5-D attention-MIL model

A `timm` CoAtNet backbone encodes every 3-slice 2.5-D window (slices `[i-1, i, i+1]` stacked as RGB-like channels) from every view into one feature vector. A **per-finding masked-softmax attention head** (one attention distribution per label, over every slice-window x view token, masked to missing views) pools those tokens into 12 study-level logits -- this lets Synovitis attend to different slices/views than ACL, instead of forcing one mean-pooled vector to serve every finding equally.

In [6]:
def make_backbone():
    """Builds the CoAtNet architecture with pretrained=False -- this notebook never calls the
    internet, including at backbone-creation time. If BACKBONE_INIT_PATH (section 1) found a local
    offline-downloaded weights file, it's loaded here with strict=False so it works whether that
    file is a bare backbone state dict or a full checkpoint that happens to contain backbone.* keys.
    With nothing attached, the backbone trains from random init -- weaker, but fully valid and
    still produces a real submission."""
    import timm
    m, name = None, None
    # smallest first -- coatnet_rmlp_2_rw_384 OOMs on a 14-16 GB T4 even at batch=1 with full
    # fine-tuning; coatnet_0_rw_224 (~25M params, native 224px) fits comfortably
    for candidate in ["coatnet_0_rw_224", "coatnet_1_rw_224", "coatnet_rmlp_1_rw_224", "coatnet_rmlp_2_rw_384"]:
        try:
            m = timm.create_model(candidate, pretrained=False, num_classes=0, global_pool="avg")
            name = candidate
            break
        except Exception as e:
            print(f"  {candidate} unavailable in this timm install ({e}); trying a smaller variant...")
    if m is None:
        raise RuntimeError("No CoAtNet variant available in this timm install. Check the attached "
                            "timm version -- CoAtNet variants need timm >= 0.9.")
    print(f"Backbone: {name} (feature dim {m.num_features}, random init -- no internet used)")
    if hasattr(m, "set_grad_checkpointing"):
        try:
            m.set_grad_checkpointing(True)   # trade compute for memory -- recomputes activations
            print("  Gradient checkpointing enabled on backbone.")
        except Exception:
            pass
    if BACKBONE_INIT_PATH:
        try:
            sd = torch.load(BACKBONE_INIT_PATH, map_location="cpu")
            if isinstance(sd, dict) and "state_dict" in sd:
                sd = sd["state_dict"]
            # accept either a bare backbone dict or a full-model dict with a 'backbone.' prefix
            sd = {k[len("backbone."):] if k.startswith("backbone.") else k: v for k, v in sd.items()}
            missing, unexpected = m.load_state_dict(sd, strict=False)
            loaded = len(m.state_dict()) - len(missing)
            print(f"  Loaded local backbone weights from {BACKBONE_INIT_PATH}: "
                  f"{loaded}/{len(m.state_dict())} tensors matched.")
        except Exception as e:
            print(f"  Could not load BACKBONE_INIT_PATH ({e}) -- keeping random init.")
    return m, name

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

class CoAtNetMIL(nn.Module):
    def __init__(self, n_labels=len(LABEL_COLS), drop=0.3):
        super().__init__()
        self.backbone, self.backbone_name = make_backbone()
        D = self.backbone.num_features
        self.view_emb = nn.Parameter(torch.zeros(len(VIEWS), D))
        self.attn = nn.Sequential(nn.Linear(D, 256), nn.GELU(), nn.Linear(256, n_labels))
        self.drop = nn.Dropout(drop)
        self.cls_w = nn.Parameter(torch.randn(n_labels, D) * 0.01)
        self.cls_b = nn.Parameter(torch.zeros(n_labels))
        self.register_buffer("img_mean", IMAGENET_MEAN)
        self.register_buffer("img_std", IMAGENET_STD)
        self._frozen = False

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad_(False)
        self.backbone.eval()
        self._frozen = True

    def unfreeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad_(True)
        self._frozen = False

    def train(self, mode=True):
        super().train(mode)
        if self._frozen:
            self.backbone.eval()
        return self

    @staticmethod
    def to_25d_windows(x):
        """x: (B, V, S, H, W) in [0,1] -> (B, V, S, 3, H, W) of overlapping 3-slice windows."""
        prev = torch.cat([x[:, :, :1], x[:, :, :-1]], 2)
        nxt = torch.cat([x[:, :, 1:], x[:, :, -1:]], 2)
        return torch.stack([prev, x, nxt], 3)

    ENCODE_CHUNK = 6   # images per backbone call during training -- bounds peak memory regardless
                        # of B*V*S; lower this further (e.g. 3) if OOM persists at BATCH=1

    def _encode(self, x_flat):
        """Runs the backbone over x_flat in small chunks, each under grad-checkpointing when
        training -- this is the actual OOM fix: it bounds peak activation memory to one chunk's
        worth regardless of how many images (B*V*S) a study expands to."""
        outs = []
        for i in range(0, x_flat.shape[0], self.ENCODE_CHUNK):
            chunk = x_flat[i:i + self.ENCODE_CHUNK]
            if self._frozen:
                # Stage B: backbone is frozen, only the head trains -- skip storing its
                # activations entirely instead of just checkpointing them
                with torch.no_grad():
                    outs.append(self.backbone(chunk))
            elif self.training and torch.is_grad_enabled():
                outs.append(torch.utils.checkpoint.checkpoint(self.backbone, chunk, use_reentrant=False))
            else:
                outs.append(self.backbone(chunk))
        return torch.cat(outs, dim=0)

    def forward(self, x, view_mask):
        B, V, S, H, W = x.shape
        x = self.to_25d_windows(x)                        # (B, V, S, 3, H, W)
        x_flat = x.reshape(B * V * S, 3, H, W)
        x_flat = (x_flat - self.img_mean) / self.img_std
        feat = self._encode(x_flat)                         # (B*V*S, D) -- chunked + checkpointed
        D = feat.shape[-1]
        f = feat.reshape(B, V, S, D) + self.view_emb[None, :, None, :]
        tok = f.reshape(B, V * S, D)
        tm = view_mask[:, :, None].expand(B, V, S).reshape(B, V * S)
        a = self.attn(tok).float().masked_fill(tm[..., None] == 0, -1e4)   # (B, V*S, n_labels)
        a = torch.softmax(a, dim=1)
        pooled = torch.einsum("btl,btd->bld", a, self.drop(tok).float())   # (B, n_labels, D)
        return (pooled * self.cls_w).sum(-1) + self.cls_b

def try_load_checkpoint(model, path):
    """Best-effort, fully offline: loads a local checkpoint's state dict if its keys line up with
    this model. Returns True on a usable (even partial, >50% of tensors matching by name+shape) load."""
    try:
        sd = torch.load(path, map_location="cpu")
        if isinstance(sd, dict) and "state_dict" in sd:
            sd = sd["state_dict"]
        own = model.state_dict()
        matched = {k: v for k, v in sd.items() if k in own and own[k].shape == v.shape}
        if len(matched) < 0.5 * len(own):
            print(f"  {os.path.basename(path)}: only {len(matched)}/{len(own)} tensors match -- skipping.")
            return False
        own.update(matched)
        model.load_state_dict(own)
        print(f"  {os.path.basename(path)}: loaded {len(matched)}/{len(own)} tensors.")
        return True
    except Exception as e:
        print(f"  {os.path.basename(path)}: failed to load ({e}) -- skipping.")
        return False


## 6. Dataset wrapper

In [7]:
class KneeStudyDataset(Dataset):
    def __init__(self, df, split="train", series_lookup=None, weights_df=None):
        self.df = df.reset_index(drop=True)
        self.split = split
        self.series_lookup = series_lookup
        # optional per-study, per-label loss weights (confidence from the attached label table);
        # aligned to self.df by StudyInstanceUID so row order never has to match
        self.weights_df = (weights_df.set_index("StudyInstanceUID") if weights_df is not None else None)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        uid = row["StudyInstanceUID"]
        rows = self.series_lookup.get(uid, EMPTY_SERIES)
        vol, mask = build_study_volume(self.split, uid, rows)
        vol_t = torch.from_numpy(vol)
        mask_t = torch.from_numpy(mask)
        if self.split == "train":
            label = torch.tensor(row[LABEL_COLS].values.astype(np.float32))
            if self.weights_df is not None:
                w = torch.tensor(self.weights_df.loc[uid, LABEL_COLS].values.astype(np.float32))
            else:
                w = torch.ones(len(LABEL_COLS), dtype=torch.float32)
            return vol_t, mask_t, label, w
        return vol_t, mask_t, uid

ser_train = {k: g for k, g in train_series.groupby("StudyInstanceUID")}
ser_test = {k: g for k, g in test_series.groupby("StudyInstanceUID")}
EMPTY_SERIES = train_series.iloc[0:0]


In [8]:
import os

os.listdir('/kaggle/input/notebooks')

['soumochatterjee']

## 7. Build the training table (gold + weak), capped to fit the time budget

In [9]:
gold_train = gold_df[["StudyInstanceUID"] + LABEL_COLS].copy()
gold_train["is_gold"] = 1

weak_train = weak_label_df.copy()
weak_train["is_gold"] = 0

train_labels_df = pd.concat([gold_train, weak_train], ignore_index=True)

# cap the weak pool so a from-scratch arm trains within a sane slice of the time budget;
# raise this once you've confirmed per-study decode+train time on real Kaggle hardware
MAX_WEAK_STUDIES = 2000
n_gold = int((train_labels_df["is_gold"] == 1).sum())
if (train_labels_df["is_gold"] == 0).sum() > MAX_WEAK_STUDIES:
    gold_rows = train_labels_df[train_labels_df["is_gold"] == 1]
    weak_rows = train_labels_df[train_labels_df["is_gold"] == 0].sample(n=MAX_WEAK_STUDIES, random_state=SEED)
    train_labels_df = pd.concat([gold_rows, weak_rows]).sample(frac=1, random_state=SEED).reset_index(drop=True)

# per-sample, per-label loss weight: 1.0 for every gold study; the attached table's own
# confidence for weak studies when available, else a flat 0.6 (weak label, unknown confidence)
if weak_conf_df is not None:
    conf_lookup = weak_conf_df.set_index("StudyInstanceUID")[LABEL_COLS]
    train_weights_df = train_labels_df[["StudyInstanceUID"]].join(
        conf_lookup.reindex(train_labels_df["StudyInstanceUID"]).reset_index(drop=True)
    )
    train_weights_df[LABEL_COLS] = train_weights_df[LABEL_COLS].fillna(0.6)
else:
    train_weights_df = train_labels_df[["StudyInstanceUID"]].copy()
    for c in LABEL_COLS:
        train_weights_df[c] = 1.0
train_weights_df.loc[train_labels_df["is_gold"].values == 1, LABEL_COLS] = 1.0

print(f"Training table: {len(train_labels_df)} studies "
      f"({n_gold} gold, {len(train_labels_df) - n_gold} weak)")
print(f"Mean per-label sample weight (weak rows): "
      f"{train_weights_df.loc[train_labels_df['is_gold'].values == 0, LABEL_COLS].values.mean():.3f}")


Training table: 2058 studies (58 gold, 2000 weak)
Mean per-label sample weight (weak rows): 0.660


## 8. Train one CoAtNet-MIL arm (time-boxed, gold-stratified holdout for monitoring)

Two-stage, same pattern that worked best across this notebook's earlier experiments: Stage A fine-tunes the whole backbone on gold+weak at a small LR; Stage B freezes the backbone and fine-tunes only the attention head on gold alone at a higher LR, which is safer with only ~50 clean examples.

In [10]:
BATCH = 4
NUM_WORKERS = min(4, os.cpu_count() or 1)
STAGE_A_EPOCHS = 6
STAGE_B_EPOCHS = 6
LR_BACKBONE = 2e-5
LR_HEAD = 1e-3
LR_STAGE_B = 2e-4
SAFETY_MARGIN_SEC = 30 * 60   # always keep enough time for inference + writing submission.csv

rng = np.random.RandomState(SEED)
gold_idx = np.where(train_labels_df["is_gold"].values == 1)[0]
weak_idx = np.where(train_labels_df["is_gold"].values == 0)[0]
val_n = max(1, int(0.2 * len(gold_idx)))
val_gold = rng.choice(gold_idx, size=val_n, replace=False)
train_gold = np.setdiff1d(gold_idx, val_gold)

def make_loader(idx, shuffle, batch=BATCH):
    ds = KneeStudyDataset(train_labels_df.iloc[idx], split="train", series_lookup=ser_train,
                           weights_df=train_weights_df)
    dl = shuffle and (len(idx) // batch >= 1)
    return DataLoader(ds, batch_size=batch, shuffle=shuffle, num_workers=NUM_WORKERS,
                       pin_memory=True, drop_last=dl)

def safe_auc(y, s):
    return roc_auc_score(y, s) if len(np.unique(y)) == 2 else np.nan

def macro_auc(y, p):
    return np.nanmean([safe_auc(y[:, j], p[:, j]) for j in range(y.shape[1])])

@torch.no_grad()
def predict(model, loader, tta_flip=True):
    model.eval(); out = []
    for batch in loader:
        x, m = batch[0].to(DEVICE), batch[1].to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            lg = model(x, m)
            if tta_flip:
                lg = (lg + model(torch.flip(x, dims=[-1]), m)) / 2
        out.append(torch.sigmoid(lg.float()).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, len(LABEL_COLS)))

def weighted_bce(logits, targets, weights):
    """BCE per-element, scaled by the attached table's confidence (or 1.0 for gold/no-confidence
    rows) -- a low-confidence weak label contributes less gradient than a clean one."""
    loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    return (loss * weights).sum() / weights.sum().clamp_min(1e-6)

trained_arm_path = None
if time_left() > SAFETY_MARGIN_SEC + 10 * 60:
    stageA_idx = np.concatenate([weak_idx, train_gold])
    train_ld_A = make_loader(stageA_idx, True)
    val_ld = make_loader(val_gold, False)

    model = CoAtNetMIL().to(DEVICE)
    ckpt_path = f"{WORK_DIR}/coatnet_mil_arm.pt"
    best_auc = -1

    opt = torch.optim.AdamW([
        {"params": model.backbone.parameters(), "lr": LR_BACKBONE},
        {"params": [p for n, p in model.named_parameters() if not n.startswith("backbone.")], "lr": LR_HEAD},
    ], weight_decay=1e-2)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

    for ep in range(STAGE_A_EPOCHS):
        if time_left() < SAFETY_MARGIN_SEC:
            print("Time budget nearly spent -- stopping Stage A early."); break
        model.train(); run = 0.0
        for x, m, y, w in tqdm(train_ld_A, desc=f"stageA epoch{ep + 1}/{STAGE_A_EPOCHS}"):
            x, m, y, w = x.to(DEVICE), m.to(DEVICE), y.to(DEVICE), w.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
                lg = model(x, m)
            loss = weighted_bce(lg.float(), y, w)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); run += loss.item()
            if time_left() < SAFETY_MARGIN_SEC:
                break
        val_auc = macro_auc(train_labels_df.iloc[val_gold][LABEL_COLS].values.astype(int), predict(model, val_ld))
        print(f"stageA ep{ep + 1} loss={run / max(1, len(train_ld_A)):.4f} gold_val_auc={val_auc:.4f} | {time_left_str()}")
        if np.nan_to_num(val_auc) >= best_auc:
            best_auc = np.nan_to_num(val_auc); torch.save(model.state_dict(), ckpt_path)

    model.freeze_backbone()
    train_ld_B = make_loader(train_gold, True, batch=min(BATCH, max(2, len(train_gold))))
    opt_b = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR_STAGE_B, weight_decay=1e-2)
    for ep in range(STAGE_B_EPOCHS):
        if time_left() < SAFETY_MARGIN_SEC:
            print("Time budget nearly spent -- stopping Stage B early."); break
        model.train(); run = 0.0
        for x, m, y, w in train_ld_B:
            x, m, y, w = x.to(DEVICE), m.to(DEVICE), y.to(DEVICE), w.to(DEVICE)
            opt_b.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
                lg = model(x, m)
            loss = weighted_bce(lg.float(), y, w)
            scaler.scale(loss).backward(); scaler.unscale_(opt_b)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt_b); scaler.update(); run += loss.item()
            if time_left() < SAFETY_MARGIN_SEC:
                break
        val_auc = macro_auc(train_labels_df.iloc[val_gold][LABEL_COLS].values.astype(int), predict(model, val_ld))
        print(f"stageB ep{ep + 1} loss={run / max(1, len(train_ld_B)):.4f} gold_val_auc={val_auc:.4f} | {time_left_str()}")
        if np.nan_to_num(val_auc) >= best_auc:
            best_auc = np.nan_to_num(val_auc); torch.save(model.state_dict(), ckpt_path)
    model.unfreeze_backbone()

    trained_arm_path = ckpt_path
    print(f"Self-trained arm saved: {ckpt_path} (best gold_val_auc={best_auc:.4f})")
    del model; gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
else:
    print("Not enough time budget left to train a self-trained arm -- relying on attached public "
          "checkpoints only (section 1). If none were attached, the submission will fall back to 0.5.")


Backbone: coatnet_0_rw_224 (feature dim 768, random init -- no internet used)
  Gradient checkpointing enabled on backbone.
  Loaded local backbone weights from /kaggle/input/datasets/amudhana/coatnet0-imagenet/coatnet_0_rw_224.pth: 240/240 tensors matched.


stageA epoch1/6:   0%|          | 0/511 [00:00<?, ?it/s]

stageA ep1 loss=0.6322 gold_val_auc=0.7310 | 441.1 min left


stageA epoch2/6:   0%|          | 0/511 [00:00<?, ?it/s]

stageA ep2 loss=0.6004 gold_val_auc=0.8090 | 402.0 min left


...

stageA ep4 loss=0.5474 gold_val_auc=0.7764 | 334.5 min left


stageA epoch5/6:   0%|          | 0/511 [00:00<?, ?it/s]

stageA ep5 loss=0.5179 gold_val_auc=0.8114 | 301.6 min left


stageA epoch6/6:   0%|          | 0/511 [00:00<?, ?it/s]

stageA ep6 loss=0.4869 gold_val_auc=0.7600 | 273.9 min left
stageB ep1 loss=0.3027 gold_val_auc=0.7510 | 273.4 min left
stageB ep2 loss=0.2831 gold_val_auc=0.7475 | 273.2 min left
stageB ep3 loss=0.2542 gold_val_auc=0.7475 | 273.0 min left
stageB ep4 loss=0.2503 gold_val_auc=0.7530 | 272.9 min left
stageB ep5 loss=0.2292 gold_val_auc=0.7586 | 272.7 min left
stageB ep6 loss=0.2112 gold_val_auc=0.7637 | 272.5 min left
Self-trained arm saved: /kaggle/working/coatnet_mil_arm.pt (best gold_val_auc=0.8364)


## 9. Assemble every usable arm and run inference

Each arm (the self-trained model above, plus every attached checkpoint that loaded cleanly) runs inference independently. Arms are combined by **rank-mean fusion**: each arm's predictions are converted to within-arm percentile ranks before averaging, which is robust to arms being calibrated on different scales -- exactly how the strongest public ensembles for this competition combine heterogeneous checkpoints.

In [11]:
test_ld = DataLoader(KneeStudyDataset(test, split="test", series_lookup=ser_test),
                     batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

arm_predictions = []   # list of (name, (n_test, 12) array)

def run_arm(name, ckpt_path):
    if time_left() < 5 * 60:
        print(f"Skipping arm '{name}': time budget exhausted."); return
    try:
        m = CoAtNetMIL().to(DEVICE)
    except Exception as e:
        print(f"Skipping arm '{name}': could not build backbone ({e})."); return
    if ckpt_path and not try_load_checkpoint(m, ckpt_path):
        print(f"Skipping arm '{name}': checkpoint did not load."); del m; return
    preds = predict(m, test_ld, tta_flip=True)
    if preds.shape[0] == len(test) and np.nanstd(preds, axis=0).max() > 1e-6:
        arm_predictions.append((name, preds))
        print(f"Arm '{name}': OK, {preds.shape}")
    else:
        print(f"Skipping arm '{name}': constant or malformed output.")
    del m; gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

if trained_arm_path:
    run_arm("self_trained", trained_arm_path)
for p in CHECKPOINT_PATHS:
    run_arm(os.path.basename(p), p)

print(f"\n{len(arm_predictions)} usable arm(s): {[n for n, _ in arm_predictions]}")


Backbone: coatnet_0_rw_224 (feature dim 768, random init -- no internet used)
  Gradient checkpointing enabled on backbone.
  Loaded local backbone weights from /kaggle/input/datasets/amudhana/coatnet0-imagenet/coatnet_0_rw_224.pth: 240/240 tensors matched.
  coatnet_mil_arm.pt: loaded 249/249 tensors.
Arm 'self_trained': OK, (3, 12)
Backbone: coatnet_0_rw_224 (feature dim 768, random init -- no internet used)
  Gradient checkpointing enabled on backbone.
  Loaded local backbone weights from /kaggle/input/datasets/amudhana/coatnet0-imagenet/coatnet_0_rw_224.pth: 240/240 tensors matched.
  coatnet_0_rw_224.pth: only 0/249 tensors match -- skipping.
Skipping arm 'coatnet_0_rw_224.pth': checkpoint did not load.

1 usable arm(s): ['self_trained']


## 10. Rank-mean ensemble -> `submission.csv`

In [12]:
def rank_pct(x):
    order = np.argsort(x, axis=0)
    ranks = np.empty_like(order, dtype=np.float32)
    n = x.shape[0]
    for j in range(x.shape[1]):
        ranks[order[:, j], j] = np.arange(n, dtype=np.float32)
    return ranks / max(n - 1, 1)

if arm_predictions:
    ranked = [rank_pct(p) for _, p in arm_predictions]
    final_pred = np.mean(ranked, axis=0)
else:
    print("No usable arms -- falling back to the 0.5 benchmark for every prediction.")
    final_pred = np.full((len(test), len(LABEL_COLS)), 0.5, dtype=np.float32)

pred_df = pd.DataFrame(final_pred, columns=LABEL_COLS)
pred_df.insert(0, "StudyInstanceUID", test["StudyInstanceUID"].values)

submission = test[["StudyInstanceUID"]].merge(pred_df, on="StudyInstanceUID", how="left")
submission[LABEL_COLS] = submission[LABEL_COLS].fillna(0.5).clip(0, 1)
submission = submission[sample_submission.columns]

assert len(submission) == len(test)
assert list(submission.columns) == list(sample_submission.columns)
submission.to_csv(f"{WORK_DIR}/submission.csv", index=False)

print(f"Wrote submission.csv: {submission.shape} | arms used: {len(arm_predictions)} | "
      f"total elapsed: {(time.time() - GLOBAL_START) / 60:.1f} min")
submission.head()


Wrote submission.csv: (3, 13) | arms used: 1 | total elapsed: 208.1 min


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.0,1.0,0.0,0.5,0.0,0.5,0.0,1.0,0.5,1.0,0.5,0.0
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.5,0.0,1.0,1.0,1.0,1.0,1.0,0.5,1.0,0.0,1.0,1.0
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,1.0,0.5,0.5,0.0,0.5,0.0,0.5,0.0,0.0,0.5,0.0,0.5


## 11. How to add the two optional inputs (do this before your real submission run)

**A. A public CoAtNet checkpoint dataset for this competition**
1. On Kaggle, go to **Datasets** and search this competition's name together with "coatnet" or "checkpoint" (several competitors have published knee-MRI CoAtNet checkpoints trained on the full corpus -- public write-ups about this competition report single checkpoints scoring 0.91-0.92 macro AUC on their own held-out gold studies, well above anything trainable from scratch in one session).
2. Open a candidate dataset, check its description for the backbone it used (ideally a CoAtNet variant -- `coatnet_rmlp_2_rw_384` or similar) and the input resolution/slice count it expects.
3. Click **+ Add Input** on this notebook and attach it. The loader in section 1 will find any `.pt`/`.pth` file automatically -- no path to edit.
4. Re-run from the top. Section 9's log tells you exactly how many tensors matched for each checkpoint; a low match count usually means a different architecture/resolution and that arm will be skipped safely rather than corrupting the ensemble.

**B. A report weak-label table (e.g. `report_labels_v2.csv`)**
1. Upload it as a Kaggle Dataset (Datasets -> New Dataset -> upload the CSV), then **+ Add Input** it on this notebook -- same as any other dataset, since this is a no-internet notebook and the file has to live on disk inside the Kaggle environment, not just in this chat.
2. Nothing else to configure: section 1 auto-detects any CSV with `StudyInstanceUID` + the 12 label columns, and section 3 also picks up `<label>__conf` columns automatically if present (exactly what `report_labels_v2.csv` has) and uses them as a per-label training weight -- a confident weak label pulls the model harder than an `UNK`/low-confidence one. `<label>__verdict` columns are ignored.

**Neither is required** -- the notebook produces a valid, internet-free submission with just the competition data attached, using its own keyword labeler and a from-scratch-trained CoAtNet-MIL arm.